In [0]:
%run ../00-common/01-Environment_Config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as f

In [0]:
if v_batch_id:
    delta_table = DeltaTable.forName(spark, control_table)

    source_df = (
        spark.createDataFrame([(v_batch_id,)], ["batch_id"])\
                .withColumn("status", f.lit("completed"))\
                .withColumn("updated_timestamp", f.current_timestamp())
    )

    delta_table.alias("trg").merge(source_df.alias("src"), "trg.batch_id = src.batch_id AND trg.status = 'in_progress'")\
                            .whenMatchedUpdate(set={
                                "status" : "src.status",
                                "updated_timestamp" : "src.updated_timestamp"
                            }).execute()
    
    print(f"Marked batch {v_batch_id} as completed")
else:
    raise Exception("Batch id id missing")

Marked batch 2025-01 as completed
